# 완성된 비즈니스 솔루션

## Day 1 프로젝트를 한 단계 업그레이드해 봅시다

### 비즈니스 과제

**회사 브로셔를 자동으로 만드는 제품**을 구현합니다.  
대상: 잠재 고객, 투자자, 채용 후보자용.

입력: 회사 이름 + 대표 웹사이트 URL.

실제 비즈니스 활용 예시는 노트북 맨 아래를 참고하세요.

문제나 아이디어가 있으면 언제든 연락 주세요!

In [3]:
# 임포트
# 실패하면 (llms) 등 '활성화된' 환경에서 실행 중인지 확인하세요

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# 초기화 및 상수

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API 키가 정상적으로 설정된 것 같습니다")
else:
    print("API 키에 문제가 있을 수 있습니다. 트러블슈팅 노트북을 확인해 보세요!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API 키가 정상적으로 설정된 것 같습니다


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## 1단계: GPT-5-nano가 관련 링크를 골라내기

### gpt-5-nano를 호출해 웹페이지의 링크를 읽고, 구조화된 JSON으로 응답하게 합니다.  
관련 있는 링크만 선별하고, "/about" 같은 상대 경로를 "https://company.com/about" 형태의 전체 URL로 바꿉니다.  
**원샷 프롬핑(one shot prompting)**을 사용합니다. 프롬프트에 원하는 응답 형식 예시를 넣어 주는 방식입니다.

LLM에게 잘 맞는 작업입니다. 미묘한 판단이 필요해서요. 웹페이지를 파싱·분석만으로 이걸 코드로 구현하려면 상당히 어렵습니다.

참고: 응답 형식을 스펙으로 강제하는 **"Structured Outputs"** 기법도 있습니다. 8주차 자율 에이전트 프로젝트에서 다룹니다.

In [6]:
link_system_prompt = """
웹페이지에서 찾은 링크 목록이 주어집니다.
회사 브로셔에 넣기에 가장 관련 있는 링크를 골라 주세요.
예: 회사 소개(About), 회사 정보(Company), 채용(Careers/Jobs) 페이지 등.
아래 예시처럼 JSON 형식으로만 응답하세요.

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
다음은 웹사이트 {url} 에서 찾은 링크 목록입니다.
회사 브로셔에 쓸 만한 관련 링크만 골라 주세요.
전체 https URL을 포함해 JSON 형식으로 응답하세요.
이용약관, 개인정보처리방침, 이메일 링크는 제외하세요.

링크 (일부는 상대 경로일 수 있음):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


다음은 웹사이트 https://edwarddonner.com 에서 찾은 링크 목록입니다.
회사 브로셔에 쓸 만한 관련 링크만 골라 주세요.
전체 https URL을 포함해 JSON 형식으로 응답하세요.
이용약관, 개인정보처리방침, 이메일 링크는 제외하세요.

링크 (일부는 상대 경로일 수 있음):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills / proficiency',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def select_relevant_links(url):
    print(f"{url} 에서 관련 링크 선별 중 ({MODEL} 호출)")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"관련 링크 {len(links['links'])}개 발견")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

https://edwarddonner.com 에서 관련 링크 선별 중 (gpt-5-nano 호출)
관련 링크 10개 발견


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'brand page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
select_relevant_links("https://huggingface.co")

https://huggingface.co 에서 관련 링크 선별 중 (gpt-5-nano 호출)
관련 링크 10개 발견


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter/X', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Discussion forum', 'url': 'https://discuss.huggingface.co'}]}

## 2단계: 브로셔 만들기

수집한 내용을 하나로 모아 GPT-5-nano에 넘기는 프롬프트를 구성합니다.

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## 랜딩 페이지:\n\n{contents}\n## 관련 링크:\n"
    for link in relevant_links['links']:
        result += f"\n\n### 링크: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

https://huggingface.co 에서 관련 링크 선별 중 (gpt-5-nano 호출)
관련 링크 8개 발견
## 랜딩 페이지:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
2 days ago
•
481k
•
744
Qwen/Qwen3.5-27B
Updated
5 days ago
•
218k
•
474
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
2 days ago
•
433k
•
406
Qwen/Qwen3.5-122B-A10B
Updated
5 days ago
•
127k
•
358
Qwen/Qwen3.5-397B-A17B
Updated
6 days ago
•
1.03M
•
1.15k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.76k
Qwen Image Multiple Angles 3D Camera
🎥
1.76k
Change the camera angle of a photo with AI
Running
on
Zero
Featured
253
Omni Video Factory
🏆
253
text to video, image to video, video extend
Running
on
Zero
MCP
1.02k
Wan2.2 14B Preview
🐌
1.02k
generat

In [16]:
brochure_system_prompt = """
회사 웹사이트의 여러 관련 페이지 내용을 분석해
잠재 고객·투자자·채용 후보를 위한 짧은 브로셔를 작성하는 어시스턴트입니다.
코드 블록 없이 마크다운으로만 응답하세요.
회사 문화, 고객, 채용/일자리 정보가 있으면 포함하세요.
"""

# 아래를 주석 해제하면 유머러스한 톤의 브로셔를 만들 수 있습니다. '톤' 조절이 얼마나 쉬운지 보여줍니다:

# brochure_system_prompt = """
# 회사 웹사이트의 여러 관련 페이지를 분석해
# 잠재 고객·투자자·채용 후보를 위한 짧고 유머러스하고 재미있는 브로셔를 작성하는 어시스턴트입니다.
# 코드 블록 없이 마크다운으로만 응답하세요.
# 회사 문화, 고객, 채용/일자리 정보가 있으면 포함하세요.
# """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
다음은 회사 '{company_name}' 의 랜딩 페이지와 관련 페이지 내용입니다.
이 정보를 바탕으로 코드 블록 없이 마크다운으로 짧은 브로셔를 작성해 주세요.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # 5,000자 초과 시 잘라냄
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

https://huggingface.co 에서 관련 링크 선별 중 (gpt-5-nano 호출)
관련 링크 12개 발견


"\n다음은 회사 'HuggingFace' 의 랜딩 페이지와 관련 페이지 내용입니다.\n이 정보를 바탕으로 코드 블록 없이 마크다운으로 짧은 브로셔를 작성해 주세요.\n\n\n## 랜딩 페이지:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n2 days ago\n•\n481k\n•\n744\nQwen/Qwen3.5-27B\nUpdated\n5 days ago\n•\n218k\n•\n474\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n2 days ago\n•\n433k\n•\n406\nQwen/Qwen3.5-122B-A10B\nUpdated\n5 days ago\n•\n127k\n•\n358\nQwen/Qwen3.5-397B-A17B\nUpdated\n6 days ago\n•\n1.03M\n•\n1.15k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.76k\nQwen Image Multiple Angles 3D Camera\n🎥\n1.76k\nChange the camera angle of a photo with AI\nRunning\non\nZero\nFeatured\n253\nOmni Video Factory\n🏆\n

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

https://huggingface.co 에서 관련 링크 선별 중 (gpt-5-nano 호출)
관련 링크 15개 발견


# Hugging Face 소개 브로셔

---

## Hugging Face  
**The AI community building the future.**  

Hugging Face는 머신러닝 커뮤니티가 함께 모여 모델, 데이터셋, 애플리케이션을 만들고 공유하며 협업하는 플랫폼입니다.   

---

## 우리의 핵심 가치 및 서비스  

- **광범위한 AI 모델 공유**  
  2백만 개 이상의 공개 AI 모델을 호스팅하여, 텍스트, 이미지, 영상, 오디오, 3D 등 다양한 형태의 데이터에 적용 가능한 혁신적인 AI 기능을 제공  
- **데이터셋 저장소**  
  50만 개 이상의 데이터셋을 통해 연구자와 개발자가 다양한 프로젝트에 필요한 고품질 데이터 사용 가능  
- **Spaces: AI 애플리케이션 개발 및 배포**  
  누구나 AI 기반 앱을 쉽게 만들고 웹에서 실행할 수 있는 독창적인 환경 제공  
- **커뮤니티 중심 협업**  
  글로벌 머신러닝 전문가와 실무진들이 지식과 리소스를 나누며 함께 발전하는 열린 플랫폼  
- **지원하는 주요 라이브러리 및 인프라**  
  PyTorch, TensorFlow, JAX, ONNX 등 인기 프레임워크와 함께 다양한 추론(실행) 프로바이더 지원  
- **기업과 맞춤형 솔루션 제공**  
  빠르고 효율적인 AI 추진을 위한 엔터프라이즈 서비스도 제공  

---

## 고객을 위한 혜택  

- AI 연구 및 제품 개발 속도 가속  
- 최신 대형 언어 모델 및 멀티모달 모델을 즉시 활용할 수 있는 유연성  
- 방대한 공개 데이터와 모델을 통해 혁신적인 솔루션을 공동 창출  
- 강력한 커뮤니티 네트워크로 지속적인 기술 지원과 협력 기회 제공  

---

## 채용 및 회사 문화  

Hugging Face는 혁신적이고 협력적인 문화를 지향합니다.  
자율성과 창의성을 존중하며, AI 기술이 사회에 미칠 긍정적 영향에 깊은 관심을 가진 열정적인 인재를 기다립니다.  
- 다양한 역할에서 머신러닝 엔지니어, 연구원, 소프트웨어 개발자, 데이터 사이언티스트 등 채용 중  
- 전 세계 인재들과 함께 성장하며 미래 AI 생태계 구축에 기여  

[채용 정보 및 지원하기](https://huggingface.co/careers)  

---

## 함께 AI 미래를 만드세요!  

지금 가입하여 Hugging Face 커뮤니티와 함께 머신러닝 모델, 데이터셋, 그리고 AI 애플리케이션 개발의 최전선에 서보세요.

[가입하기](https://huggingface.co/join)  
[더 알아보기](https://huggingface.co)  

---

**Hugging Face**  
_The Home of Machine Learning_  
AI 혁신을 이끄는 플랫폼에서 당신의 가능성을 펼치십시오.

## 마지막: 작은 개선

조금만 수정하면 OpenAI 응답을 **스트리밍**으로 받을 수 있어서,
타이핑되는 것처럼 순차적으로 출력됩니다.

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

https://huggingface.co 에서 관련 링크 선별 중 (gpt-5-nano 호출)
관련 링크 15개 발견


# Hugging Face 소개 브로셔

---

## AI의 미래를 함께 만드는 커뮤니티, Hugging Face

Hugging Face는 머신러닝(Machine Learning) 커뮤니티를 위한 협업 플랫폼으로, 전 세계 연구자와 개발자가 모델, 데이터셋, 애플리케이션을 제작하고 공유하며 함께 발전시키고 있습니다.

- **2백만 개 이상의 AI 모델**과 50만 개 이상의 데이터셋이 공개되어 있어 자유롭게 탐색하고 실험할 수 있습니다.
- 텍스트, 이미지, 비디오, 오디오, 3D 등 다양한 형태의 AI 기술을 지원하여 폭넓은 프로젝트에 활용할 수 있습니다.
- 사용자가 직접 만든 애플리케이션을 공유하고 포트폴리오를 구축할 수 있어, 개인과 조직 모두 성장 가능한 환경을 제공합니다.

---

## 기업을 위한 맞춤형 AI 플랫폼

Hugging Face 기업용 허브는 강화된 보안, 세분화된 접근 제어, 전담 지원 체계 등 엔터프라이즈급 기능을 갖추고 있습니다.  
주요 혜택:

- **기업 단일 로그인(SSO)** 지원으로 안전하고 편리한 접속 관리
- **지역별 데이터 저장 관리** 및 **감사 로그** 제공으로 보안 및 규정 준수 강화
- **사용 현황 분석 및 예산 관리** 대시보드로 최적의 리소스 운영
- **확장 가능한 컴퓨팅 옵션**과 5배 향상된 ZeroGPU 할당으로 성능 극대화
- 멤버 별 프라이빗 저장공간 및 맞춤형 승인 정책으로 효율적인 협업 환경 제공

합리적인 팀 이용 요금과 기업 맞춤 계약 옵션으로 규모와 필요에 따라 유연하게 이용하실 수 있습니다.

---

## 커뮤니티와 함께 성장하는 혁신 문화

Hugging Face는 최첨단 기술을 탐구하는 과학자 팀과 활발한 글로벌 커뮤니티가 함께하는 열린 AI 혁신의 중심입니다.

- 누구나 쉽게 참여하고 배우며 성장할 수 있는 열린 공간을 지향합니다.
- 윤리적이고 투명한 AI 개발을 통한 더 나은 미래를 목표로 합니다.
- 활발한 포럼, 문서, 소셜 미디어 활동으로 최신 정보와 동향을 신속하게 공유합니다.

---

## 잠재 고객, 투자자, 그리고 채용 후보자 여러분께

- AI 기술 혁신을 선도하는 글로벌 플랫폼에서 귀하의 비즈니스와 커리어를 한 단계 업그레이드하세요.
- 풍부한 오픈소스 자원과 강력한 기업 지원으로 효율적인 프로젝트 수행이 가능합니다.
- 성장하는 AI 커뮤니티와 함께 협력하며 지속 가능한 발전을 이루고자 하는 분들을 환영합니다.

---

### 지금 Hugging Face와 함께 AI의 미래를 만들어 가세요!  
[웹사이트 방문하기](https://huggingface.co)

---

**Hugging Face**  
-The AI community building the future.-

In [ ]:
# Hugging Face 브로셔를 만들 때 시스템 프롬프트를 유머 버전으로 바꿔 보세요:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">비즈니스 활용</h2>
            <span style="color:#181;">이번 연습에서는 Day 1 코드를 확장해 여러 번의 LLM 호출과 문서 생성을 해봤습니다.

여러 LLM 호출을 조합한, 에이전틱 AI 설계 패턴의 첫 예시라고 할 수 있습니다. 2주차에서 더 다루고, 8주차에는 완전 자율 에이전트를 만들며 에이전틱 AI를 본격적으로 다룹니다.

이런 식의 콘텐츠 생성은 가장 흔한 사용 사례 중 하나입니다. 요약과 마찬가지로 어떤 업종에도 적용할 수 있습니다. 마케팅 문구 작성, 스펙에서 제품 튜토리얼 생성, 맞춤 이메일 작성 등 다양하게 쓸 수 있어요. 본인 비즈니스에 어떻게 적용할지 탐색해 보고, 프로토타입을 한 번 만들어 보세요. community-contributions 폴더에서 다른 수강생들이 만든 프로젝트도 참고해 보시면 좋습니다!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">2주차로 넘어가기 전에 (2주차도 정말 재밌어요)</h2>
            <span style="color:#900;">1주차 마무리 과제는 week1의 EXERCISE 노트북을 확인하세요. Frontier API를 다루는 필수 연습이 되고, 2주차 준비에도 도움이 됩니다.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">유용한 자료 3가지</h2>
            <span style="color:#f71;">1. 강의 자료는 <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">여기</a>에서 볼 수 있습니다.<br/>
            2. LinkedIn <a href="https://www.linkedin.com/in/eddonner/">프로필</a> – 수강생들과 연결되는 걸 좋아합니다!<br/>
            3. X(Twitter) <a href="https://x.com/edwarddonner">@edwarddonner</a> – 사용법을 배우는 중이에요.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">마지막으로, 부탁 하나 드립니다</h2>
            <span style="color:#090;">
                Udemy에서 이 강의에 평점을 남겨 주시면 노출에 큰 영향을 준다고 합니다. 가능하시다면 잠깐만 평점 부탁드려요, 정말 감사하겠습니다! 그 외에도 도움이 필요하시면 언제든 ed@edwarddonner.com 으로 연락 주세요.
            </span>
        </td>
    </tr>
</table>